# Create cistarget database

In [3]:
import os
import pandas as pd
import numpy as np
from pathlib import Path
import seaborn as sns
import matplotlib.pyplot as plt
import pickle

In [1]:
#root_dir = "."
#os.chdir(Path(root_dir) / "scenicplus" / "all")

In [4]:
region_names = pd.read_csv("Seurat_Data/region_names.tsv", sep="\t", header=None, index_col=0)

In [6]:
with open("LTB_consensus_peaks.bed", "w") as fout:
    for peak_name in region_names.index:
        chromosome, coords = peak_name.split("-", maxsplit=1)
        start, end = coords.split("-")
        fout.write("{}\t{}\t{}\n".format(chromosome, start, end))

In [5]:
## follow steps in the tutorial to create background fasta file

In [18]:
slurm_script_template = (
    "#!/bin/bash\n"
    "#SBATCH -p condo\n"
    "#SBATCH -q condo\n"
    "#SBATCH -J ct_{0}\n"
    "#SBATCH -N 3\n"
    "#SBATCH -c 8\n"
    "#SBATCH --mem 400G\n"
    "#SBATCH -t 12:00:00\n"
    "#SBATCH -o /tscc/nfs/home/rlancione/jeo//ct_{0}.out\n"
    "#SBATCH -e /tscc/nfs/home/rlancione/jeo//ct_{0}.err\n"
    "#SBATCH --mail-user rlancione@ucsd.edu\n"
    "#SBATCH --mail-type FAIL\n"
    "#SBATCH -A csd772\n"
    "\n"
    "set -e\n"
    "source ~/.bashrc\n"
    "conda activate scenicplus\n"
    "\n"

    "SCRIPT_DIR=\"/tscc/nfs/home/rlancione/software/create_cisTarget_databases/\"\n"
    "DATABASE_PREFIX=\"LTB\"\n"
    "OUT_DIR=\"/tscc/projects/ps-epigen/users/rlan/Liver_TB/Scenic/AllData/cistarget_db/out/\"\n"
    "CBDIR=\"/tscc/nfs/home/rlancione/misc/aertslab_motif_colleciton/v10nr_clust_public/singletons\"\n"
    "FASTA_FILE=\"/tscc/projects/ps-epigen/users/rlan/Liver_TB/Scenic/AllData/prepare_fasta/consensus.fa\"\n"
    "MOTIF_LIST=\"/tscc/projects/ps-epigen/users/rlan/Liver_TB/Scenic/AllData/cistarget_db/motifs.txt\"\n"
    "\n"
    "create_cistarget_databases_dir=\"/tscc/nfs/home/rlancione/software/create_cisTarget_databases/\"\n\n"

    "\"${{SCRIPT_DIR}}/create_cistarget_motif_databases.py\" \\\n"
        "-f ${{FASTA_FILE}} \\\n"
        "-M ${{CBDIR}} \\\n"
        "-m ${{MOTIF_LIST}} \\\n"
        "-o ${{OUT_DIR}}/partial_database/${{DATABASE_PREFIX}} \\\n"
        "-p {0} 8 \\\n"
        "--bgpadding 1000 \\\n"
        "-t 20\n"
)

In [19]:
for index in np.arange(1, 9):
    with open("cistarget_db/partial_slurm/ct_{}.sh".format(index), "w") as f:
        f.write(slurm_script_template.format(index))